In [8]:
import os, glob, json, re
import pandas as pd

In [9]:
df = pd.read_csv('data_listings/data-listings.csv', encoding='utf-8-sig')
df

,title,description,project_unit,project_name,area,floor_area,price,location,bedrooms,bathrooms,floor,condition,property_type,offer_type,construction_year,parking_spaces,ownership_type,project_link,publish_date,publish_by
0,3BR Condo Calinea Near SM Grand Central Monume...,Enjoy excellent accessibility by living in THE...,The Calinea Tower,The Calinea Tower,79 sqm,79 sqm,"₱ 11,742,000","Grace Park East, Caloocan",3 bedrooms,2 bathrooms,NaN,NaN,Condo,For Sale,NaN,NaN,NaN,https://www.lamudi.com.ph/projects/the-calinea...,3 Jul 2025,Corz Cascayan
1,2BR CondoNear SM grand Central Calinea DMCI Pr...,THE CALINEA TOWER by DMCI Homes\r\n\r\nM. H. D...,The Calinea Tower,The Calinea Tower,51 sqm,51 sqm,"₱ 9,100,000","Grace Park East, Caloocan",2 bedrooms,1 bathroom,NaN,NaN,Condo,For Sale,NaN,NaN,NaN,https://www.lamudi.com.ph/projects/the-calinea...,3 Jul 2025,Corz Cascayan
2,Calinea tower 126sqm 3BR DMCI newest Presellin...,Enjoys excellent accessibility by living in TH...,The Calinea Tower,The Calinea Tower,126 sqm,126 sqm,"₱ 22,000,000","Grace Park East, Caloocan",3 bedrooms,3 bathrooms,NaN,Unfurnished,Condo,For Sale,NaN,NaN,NaN,https://www.lamudi.com.ph/projects/the-calinea...,3 Jul 2025,Corz Cascayan
3,Spacious Studio Condo For Sale in Caloocan City,NaN,The Calinea Tower,The Calinea Tower,NaN,33 sqm,"₱ 6,346,000","Grace Park East, Caloocan",NaN,1 bathroom,NaN,Unfurnished,Condo,For Sale,NaN,NaN,Freehold,https://www.lamudi.com.ph/projects/the-calinea...,14 Jun 2025,"Michelle Madarang, REB"
4,FOR SALE: The Calinea Tower,NaN,The Calinea Tower,The Calinea Tower,NaN,86 sqm,"₱ 13,025,000","Grace Park East, Caloocan",3 bedrooms,2 bathrooms,NaN,Unfurnished,Condo,For Sale,NaN,NaN,NaN,https://www.lamudi.com.ph/projects/the-calinea...,13 Jun 2025,Condominium for Sale PH by Rey Ann Roguel
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15596,VICTORIA DE VALENZUELA Near Our Lady of Fatima...,Get a unit for as low as P8k a month!!!\r\nNew...,NaN,NaN,NaN,22 sqm,"₱ 2,709,550","Valenzuela, Metro Manila",1 bedroom,1 bathroom,45,Unfurnished,Condo,For Sale,NaN,NaN,Freehold,NaN,2 May 2022,Calvin Reyes
15597,CONDO NEAR FATIMA UNIVERSITY AND MCU MEDICAL C...,Get a unit for as low as P8k a month!!!\r\nNew...,NaN,NaN,NaN,22 sqm,"₱ 2,700,888","Valenzuela, Metro Manila",2 bedrooms,2 bathrooms,45,Unfurnished,Condo,For Sale,NaN,NaN,Freehold,NaN,2 May 2022,Calvin Reyes
15598,affordable condominium in valenzuela city near...,Victoria de Valenzuela“The Newest Sports Condo...,NaN,NaN,NaN,22 sqm,"₱ 2,750,002","Valenzuela, Metro Manila",1 bedroom,1 bathroom,45,Unfurnished,Condo,For Sale,NaN,NaN,Freehold,NaN,2 May 2022,Calvin Reyes
15599,condominium in valenzuela near fatima universi...,2.7m to 3m total price \r\n15k reservation fee...,NaN,NaN,NaN,22 sqm,"₱ 2,780,000","Valenzuela, Metro Manila",1 bedroom,1 bathroom,45,Unfurnished,Condo,For Sale,NaN,NaN,Freehold,NaN,2 May 2022,Calvin Reyes


In [10]:
# Drop 'publish_date'. Remove duplicates and reset index
df = df.drop(columns=['publish_date']).drop_duplicates().reset_index(drop=True)

# strip multiple spaces in 'title' and 'description' columns
df['title'] = df['title'].str.replace(r'\s{2,}', ' ', regex=True)
df['description'] = df['description'].str.replace(r'\s{2,}', ' ', regex=True)
df.describe(include='all')

,title,description,project_unit,project_name,area,floor_area,price,location,bedrooms,bathrooms,floor,condition,property_type,offer_type,construction_year,parking_spaces,ownership_type,project_link,publish_by
count,12107,8949,6799,6799,3018,12083,11529,12107,10350,10854,4429,9233,12107,12107,983.000000,2010.000000,4123,6799,12107
unique,10686,6674,742,741,324,409,3582,237,21,21,170,3,5,2,NaN,NaN,2,759,1035
top,Bare 2BR Unit for Sale at Mi Casa Hawaii Tower...,READY FOR OCCUPANCY CONDO IN BAY AREA 2br pet ...,The Hermosa,The Hermosa,21 sqm,23 sqm,"₱ 4,500,000","Fort Bonifacio, Taguig",1 bedroom,1 bathroom,12,Unfurnished,Condo,For Sale,NaN,NaN,Freehold,https://www.lamudi.com.ph/projects/41032-73-4f...,27C Realty
freq,65,70,283,283,100,446,163,1070,4443,7118,259,4024,10887,11445,NaN,NaN,4028,283,589
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2021.150560,1.479602,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.864497,0.900103,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1500.000000,1.000000,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020.000000,1.000000,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024.000000,1.000000,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025.000000,2.000000,NaN,NaN,NaN


In [11]:
df_projects = pd.read_csv('data_properties/data-properties-with-corrections.csv', dtype=str, encoding='utf-8-sig')
df_projects

,project_name,location
0,100 West,"Pio Del Pilar, Makati, Metro Manila"
1,100 West Makati,"Pio Del Pilar, Makati, Metro Manila"
2,1001 Parkway Residences,"Muntinlupa, Metro Manila"
3,101 Newport BLVD,"Pasay, Metro Manila"
4,101 Xavierville,"Loyola Heights, Quezon City, Metro Manila"
...,...,...
1011,dakotaresidences,"Tugatog, Malabon, Metro Manila"
1012,mckinley hill garden villas,"Bagong Tanyag, Taguig, Metro Manila"
1013,oriental gardens makati,"Bangkal, Makati, Metro Manila"
1014,symfoni kamias,"Ramon Magsaysay, Quezon City, Metro Manila"


In [ ]:
# from each row in df, if 'project_name' is empty, find the 1st matching 'project_name' in df_projects that is a substring of 'description' or 'title' and fill in 'project_name' from df_projects
for i, row in df.iterrows():
    if pd.isna(row['project_name']) or row['project_name'].strip() == '':
        for j, proj_row in df_projects.iterrows():
            if proj_row['project_name'].lower() in str(row['description']).lower():
                df.at[i, 'project_name'] = proj_row['project_name']
            else:
                if proj_row['project_name'].lower() in str(row['title']).lower():
                    df.at[i, 'project_name'] = proj_row['project_name']
                    break


,title,description,project_unit,project_name,area,floor_area,price,location,bedrooms,bathrooms,floor,condition,property_type,offer_type,construction_year,parking_spaces,ownership_type,project_link,publish_by
count,12107,8949,6799,10394,3018,12083,11529,12107,10350,10854,4429,9233,12107,12107,983.000000,2010.000000,4123,6799,12107
unique,10686,6674,742,804,324,409,3582,237,21,21,170,3,5,2,NaN,NaN,2,759,1035
top,Bare 2BR Unit for Sale at Mi Casa Hawaii Tower...,READY FOR OCCUPANCY CONDO IN BAY AREA 2br pet ...,The Hermosa,Palm Beach West,21 sqm,23 sqm,"₱ 4,500,000","Fort Bonifacio, Taguig",1 bedroom,1 bathroom,12,Unfurnished,Condo,For Sale,NaN,NaN,Freehold,https://www.lamudi.com.ph/projects/41032-73-4f...,27C Realty
freq,65,70,283,369,100,446,163,1070,4443,7118,259,4024,10887,11445,NaN,NaN,4028,283,589
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2021.150560,1.479602,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.864497,0.900103,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1500.000000,1.000000,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020.000000,1.000000,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024.000000,1.000000,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025.000000,2.000000,NaN,NaN,NaN


In [58]:
df1 = df.copy()

df1.describe(include='all')

,title,description,project_unit,project_name,area,floor_area,price,location,bedrooms,bathrooms,floor,condition,property_type,offer_type,construction_year,parking_spaces,ownership_type,project_link,publish_by
count,12107,8949,6799,10394,3018,12083,11529,12107,10350,10854,4429,9233,12107,12107,983.000000,2010.000000,4123,6799,12107
unique,10686,6674,742,804,324,409,3582,237,21,21,170,3,5,2,NaN,NaN,2,759,1035
top,Bare 2BR Unit for Sale at Mi Casa Hawaii Tower...,READY FOR OCCUPANCY CONDO IN BAY AREA 2br pet ...,The Hermosa,Palm Beach West,21 sqm,23 sqm,"₱ 4,500,000","Fort Bonifacio, Taguig",1 bedroom,1 bathroom,12,Unfurnished,Condo,For Sale,NaN,NaN,Freehold,https://www.lamudi.com.ph/projects/41032-73-4f...,27C Realty
freq,65,70,283,369,100,446,163,1070,4443,7118,259,4024,10887,11445,NaN,NaN,4028,283,589
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2021.150560,1.479602,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.864497,0.900103,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1500.000000,1.000000,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020.000000,1.000000,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024.000000,1.000000,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025.000000,2.000000,NaN,NaN,NaN


In [59]:
# remove duplicate rows and filter out rows with missing 'project_name', 'price'
df1 = df1.drop_duplicates().dropna(subset=['project_name', 'price', 'floor_area', 'bedrooms', 'bathrooms', 'condition']).reset_index(drop=True)
df1.describe(include='all')

,title,description,project_unit,project_name,area,floor_area,price,location,bedrooms,bathrooms,floor,condition,property_type,offer_type,construction_year,parking_spaces,ownership_type,project_link,publish_by
count,6774,5199,4819,6774,2146,6774,6774,6774,6774,6774,3005,6774,6774,6774,682.000000,1440.000000,2667,4819,6774
unique,5969,3993,648,705,296,345,2444,195,10,14,129,3,5,2,NaN,NaN,2,660,764
top,Bare 2BR Unit for Sale at Mi Casa Hawaii Tower...,READY FOR OCCUPANCY CONDO IN BAY AREA 2br pet ...,The Hermosa,Sonora Garden Residences,28 sqm,38 sqm,"₱ 4,500,000","Fort Bonifacio, Taguig",1 bedroom,1 bathroom,12,Unfurnished,Condo,For Sale,NaN,NaN,Freehold,https://www.lamudi.com.ph/projects/41032-73-4f...,27C Realty
freq,65,69,131,185,36,226,90,712,2998,4300,166,2944,6103,6765,NaN,NaN,2613,131,540
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2021.246334,1.484722,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.518314,0.751955,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1987.000000,1.000000,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019.000000,1.000000,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024.000000,1.000000,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025.000000,2.000000,NaN,NaN,NaN


In [60]:
# remove non-numeric characters from 'price', 'area', 'floor_area', 'bedrooms', 'bathrooms' columns and convert to numeric dtype
df1['price'] = pd.to_numeric(df1['price'].str.replace(r'[^\d.]*', '', regex=True), errors='coerce')
df1['area'] = pd.to_numeric(df1['area'].str.replace(r'[^\d.]*', '', regex=True), errors='coerce')
df1['floor_area'] = pd.to_numeric(df1['floor_area'].str.replace(r'[^\d.]*', '', regex=True), errors='coerce')
df1['bedrooms'] = pd.to_numeric(df1['bedrooms'].str.replace(r'[^\d.]*', '', regex=True), errors='coerce')
df1['bathrooms'] = pd.to_numeric(df1['bathrooms'].str.replace(r'[^\d.]*', '', regex=True), errors='coerce')

df1.describe(include='all')

,title,description,project_unit,project_name,area,floor_area,price,location,bedrooms,bathrooms,floor,condition,property_type,offer_type,construction_year,parking_spaces,ownership_type,project_link,publish_by
count,6774,5199,4819,6774,2146.000000,6.774000e+03,6.770000e+03,6774,6774.000000,6774.000000,3005,6774,6774,6774,682.000000,1440.000000,2667,4819,6774
unique,5969,3993,648,705,NaN,NaN,NaN,195,NaN,NaN,129,3,5,2,NaN,NaN,2,660,764
top,Bare 2BR Unit for Sale at Mi Casa Hawaii Tower...,READY FOR OCCUPANCY CONDO IN BAY AREA 2br pet ...,The Hermosa,Sonora Garden Residences,NaN,NaN,NaN,"Fort Bonifacio, Taguig",NaN,NaN,12,Unfurnished,Condo,For Sale,NaN,NaN,Freehold,https://www.lamudi.com.ph/projects/41032-73-4f...,27C Realty
freq,65,69,131,185,NaN,NaN,NaN,712,NaN,NaN,166,2944,6103,6765,NaN,NaN,2613,131,540
mean,NaN,NaN,NaN,NaN,147.862069,1.091452e+04,2.204859e+07,NaN,1.773546,1.572040,NaN,NaN,NaN,NaN,2021.246334,1.484722,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,1499.748716,6.523691e+05,3.763642e+07,NaN,1.007913,1.057504,NaN,NaN,NaN,NaN,6.518314,0.751955,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,2.000000,1.000000e+00,1.700000e+01,NaN,1.000000,1.000000,NaN,NaN,NaN,NaN,1987.000000,1.000000,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,47.000000,3.300000e+01,5.400000e+06,NaN,1.000000,1.000000,NaN,NaN,NaN,NaN,2019.000000,1.000000,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,77.000000,5.400000e+01,9.500000e+06,NaN,2.000000,1.000000,NaN,NaN,NaN,NaN,2024.000000,1.000000,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,135.000000,9.000000e+01,2.291000e+07,NaN,2.000000,2.000000,NaN,NaN,NaN,NaN,2025.000000,2.000000,NaN,NaN,NaN


In [61]:
# filter out rows with 'price' < 1000000 or 'floor_area' < 18
df1 = df1[(df1['price'] >= 1000000) & (df1['floor_area'] >= 18)].reset_index(drop=True)
df1.describe(include='all')

,title,description,project_unit,project_name,area,floor_area,price,location,bedrooms,bathrooms,floor,condition,property_type,offer_type,construction_year,parking_spaces,ownership_type,project_link,publish_by
count,6451,4928,4630,6451,2092.000000,6.451000e+03,6.451000e+03,6451,6451.000000,6451.000000,2773,6451,6451,6451,647.000000,1436.000000,2439,4630,6451
unique,5744,3899,642,697,NaN,NaN,NaN,192,NaN,NaN,128,3,5,2,NaN,NaN,2,654,740
top,Bare 2BR Unit for Sale at Mi Casa Hawaii Tower...,READY FOR OCCUPANCY CONDO IN BAY AREA 2br pet ...,The Hermosa,Sonora Garden Residences,NaN,NaN,NaN,"Fort Bonifacio, Taguig",NaN,NaN,12,Unfurnished,Condo,For Sale,NaN,NaN,Freehold,https://www.lamudi.com.ph/projects/41032-73-4f...,27C Realty
freq,65,69,131,183,NaN,NaN,NaN,711,NaN,NaN,140,2876,5795,6446,NaN,NaN,2387,131,539
mean,NaN,NaN,NaN,NaN,150.800191,1.145947e+04,2.311393e+07,NaN,1.795381,1.595257,NaN,NaN,NaN,NaN,2021.185471,1.486072,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,1518.875145,6.684994e+05,3.824089e+07,NaN,0.951531,1.076040,NaN,NaN,NaN,NaN,6.575325,0.752567,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,2.000000,1.800000e+01,1.000000e+06,NaN,1.000000,1.000000,NaN,NaN,NaN,NaN,1987.000000,1.000000,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,49.000000,3.400000e+01,5.850000e+06,NaN,1.000000,1.000000,NaN,NaN,NaN,NaN,2019.000000,1.000000,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,79.000000,5.600000e+01,1.000000e+07,NaN,2.000000,1.000000,NaN,NaN,NaN,NaN,2024.000000,1.000000,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,136.000000,9.500000e+01,2.380000e+07,NaN,2.000000,2.000000,NaN,NaN,NaN,NaN,2025.000000,2.000000,NaN,NaN,NaN


In [62]:
def find_with_parking(text):
    # Returns the matched string if found, else None
    match = re.search(r'with(\s+\w+)?\s+parking', str(text), re.IGNORECASE)
    if match:
        return match.group(0)
    return None

df1['parking_info'] = df1.apply(lambda row: find_with_parking(row['title']) or find_with_parking(row['description']), axis=1)

# get numeric parking count from parking_info if available, else set to 1 if parking_info is not null and 0 if parking_info is null
def extract_parking_count(parking_info):
    if pd.isna(parking_info):
        return 0
    match = re.search(r'(\d+)', parking_info)
    if match:
        return int(match.group(1))
    return 1

# get max between existing 'parking_spaces' column and extracted parking count from parking_info, add new column 'parking'
df1['parking'] = df1.apply(lambda row: max(row['parking_spaces'], extract_parking_count(row['parking_info'])) if not pd.isna(row['parking_spaces']) else extract_parking_count(row['parking_info']), axis=1)

# drop 'parking_spaces' and 'parking_info' columns
df1 = df1.drop(columns=['parking_spaces', 'parking_info'])

df1.describe(include='all')

,title,description,project_unit,project_name,area,floor_area,price,location,bedrooms,bathrooms,floor,condition,property_type,offer_type,construction_year,ownership_type,project_link,publish_by,parking
count,6451,4928,4630,6451,2092.000000,6.451000e+03,6.451000e+03,6451,6451.000000,6451.000000,2773,6451,6451,6451,647.000000,2439,4630,6451,6451.000000
unique,5744,3899,642,697,NaN,NaN,NaN,192,NaN,NaN,128,3,5,2,NaN,2,654,740,NaN
top,Bare 2BR Unit for Sale at Mi Casa Hawaii Tower...,READY FOR OCCUPANCY CONDO IN BAY AREA 2br pet ...,The Hermosa,Sonora Garden Residences,NaN,NaN,NaN,"Fort Bonifacio, Taguig",NaN,NaN,12,Unfurnished,Condo,For Sale,NaN,Freehold,https://www.lamudi.com.ph/projects/41032-73-4f...,27C Realty,NaN
freq,65,69,131,183,NaN,NaN,NaN,711,NaN,NaN,140,2876,5795,6446,NaN,2387,131,539,NaN
mean,NaN,NaN,NaN,NaN,150.800191,1.145947e+04,2.311393e+07,NaN,1.795381,1.595257,NaN,NaN,NaN,NaN,2021.185471,NaN,NaN,NaN,0.387692
std,NaN,NaN,NaN,NaN,1518.875145,6.684994e+05,3.824089e+07,NaN,0.951531,1.076040,NaN,NaN,NaN,NaN,6.575325,NaN,NaN,NaN,0.734887
min,NaN,NaN,NaN,NaN,2.000000,1.800000e+01,1.000000e+06,NaN,1.000000,1.000000,NaN,NaN,NaN,NaN,1987.000000,NaN,NaN,NaN,0.000000
25%,NaN,NaN,NaN,NaN,49.000000,3.400000e+01,5.850000e+06,NaN,1.000000,1.000000,NaN,NaN,NaN,NaN,2019.000000,NaN,NaN,NaN,0.000000
50%,NaN,NaN,NaN,NaN,79.000000,5.600000e+01,1.000000e+07,NaN,2.000000,1.000000,NaN,NaN,NaN,NaN,2024.000000,NaN,NaN,NaN,0.000000
75%,NaN,NaN,NaN,NaN,136.000000,9.500000e+01,2.380000e+07,NaN,2.000000,2.000000,NaN,NaN,NaN,NaN,2025.000000,NaN,NaN,NaN,1.000000


In [63]:
# Filter property_type to exclude "Condotel"
df1 = df1[df1['property_type'] != "Condotel"]

# Filter rows based on column: 'offer_type'
df1 = df1[df1['offer_type'] == "For Sale"]

df1.describe(include='all')

,title,description,project_unit,project_name,area,floor_area,price,location,bedrooms,bathrooms,floor,condition,property_type,offer_type,construction_year,ownership_type,project_link,publish_by,parking
count,6370,4847,4565,6370,2068.000000,6.370000e+03,6.370000e+03,6370,6370.000000,6370.000000,2737,6370,6370,6370,644.000000,2435,4565,6370,6370.000000
unique,5665,3826,637,693,NaN,NaN,NaN,192,NaN,NaN,125,3,4,1,NaN,2,649,734,NaN
top,Bare 2BR Unit for Sale at Mi Casa Hawaii Tower...,READY FOR OCCUPANCY CONDO IN BAY AREA 2br pet ...,The Hermosa,Sonora Garden Residences,NaN,NaN,NaN,"Fort Bonifacio, Taguig",NaN,NaN,12,Unfurnished,Condo,For Sale,NaN,Freehold,https://www.lamudi.com.ph/projects/41032-73-4f...,27C Realty,NaN
freq,65,69,131,183,NaN,NaN,NaN,702,NaN,NaN,140,2842,5790,6370,NaN,2383,131,529,NaN
mean,NaN,NaN,NaN,NaN,150.968569,1.160420e+04,2.315021e+07,NaN,1.793407,1.594035,NaN,NaN,NaN,NaN,2021.189441,NaN,NaN,NaN,0.388540
std,NaN,NaN,NaN,NaN,1527.626823,6.727357e+05,3.833411e+07,NaN,0.951825,1.076459,NaN,NaN,NaN,NaN,6.590073,NaN,NaN,NaN,0.732075
min,NaN,NaN,NaN,NaN,2.000000,1.800000e+01,1.000000e+06,NaN,1.000000,1.000000,NaN,NaN,NaN,NaN,1987.000000,NaN,NaN,NaN,0.000000
25%,NaN,NaN,NaN,NaN,49.000000,3.400000e+01,5.873500e+06,NaN,1.000000,1.000000,NaN,NaN,NaN,NaN,2019.000000,NaN,NaN,NaN,0.000000
50%,NaN,NaN,NaN,NaN,79.000000,5.600000e+01,1.000000e+07,NaN,2.000000,1.000000,NaN,NaN,NaN,NaN,2024.000000,NaN,NaN,NaN,0.000000
75%,NaN,NaN,NaN,NaN,136.000000,9.475000e+01,2.399992e+07,NaN,2.000000,2.000000,NaN,NaN,NaN,NaN,2025.000000,NaN,NaN,NaN,1.000000


In [64]:
# drop unnecessary columns
df1 = df1.drop(columns=['title', 'description', 'project_unit', 'location', 'area', 'floor', 'property_type', 'offer_type', 'construction_year', 'ownership_type', 'project_link', 'publish_by'])

# remove duplicate rows
df1.drop_duplicates()

df1.describe(include='all')

,project_name,floor_area,price,bedrooms,bathrooms,condition,parking
count,6370,6.370000e+03,6.370000e+03,6370.000000,6370.000000,6370,6370.000000
unique,693,NaN,NaN,NaN,NaN,3,NaN
top,Sonora Garden Residences,NaN,NaN,NaN,NaN,Unfurnished,NaN
freq,183,NaN,NaN,NaN,NaN,2842,NaN
mean,NaN,1.160420e+04,2.315021e+07,1.793407,1.594035,NaN,0.388540
std,NaN,6.727357e+05,3.833411e+07,0.951825,1.076459,NaN,0.732075
min,NaN,1.800000e+01,1.000000e+06,1.000000,1.000000,NaN,0.000000
25%,NaN,3.400000e+01,5.873500e+06,1.000000,1.000000,NaN,0.000000
50%,NaN,5.600000e+01,1.000000e+07,2.000000,1.000000,NaN,0.000000
75%,NaN,9.475000e+01,2.399992e+07,2.000000,2.000000,NaN,1.000000


In [ ]:
df1

,project_name,floor_area,price,bedrooms,bathrooms,condition,parking,price_per_sqm
0,The Calinea Tower,126,22000000.0,3,3,Unfurnished,0.0,174603.174603
1,The Calinea Tower,86,13025000.0,3,2,Unfurnished,0.0,151453.488372
2,The Calinea Tower,30,6007000.0,1,1,Unfurnished,0.0,200233.333333
3,The Calinea Tower,70,10938000.0,2,1,Unfurnished,0.0,156257.142857
4,The Calinea Tower,79,11700000.0,3,2,Unfurnished,0.0,148101.265823
...,...,...,...,...,...,...,...,...
6446,Isabelle de Valenzuela,42,5000000.0,2,1,Unfurnished,0.0,119047.619048
6447,Isabelle de Valenzuela,42,2100000.0,3,1,Unfurnished,0.0,50000.000000
6448,Alta Spatial,33,4999100.0,2,1,Partly furnished,0.0,151487.878788
6449,Alta Spatial,33,4051961.0,2,1,Unfurnished,0.0,122786.696970


In [72]:
# add 'price_per_sqm' column
df1['price_per_sqm'] = df1['price'] / df1['floor_area']
df1

,project_name,floor_area,price,bedrooms,bathrooms,condition,parking,price_per_sqm
0,The Calinea Tower,126,22000000.0,3,3,Unfurnished,0.0,174603.174603
1,The Calinea Tower,86,13025000.0,3,2,Unfurnished,0.0,151453.488372
2,The Calinea Tower,30,6007000.0,1,1,Unfurnished,0.0,200233.333333
3,The Calinea Tower,70,10938000.0,2,1,Unfurnished,0.0,156257.142857
4,The Calinea Tower,79,11700000.0,3,2,Unfurnished,0.0,148101.265823
...,...,...,...,...,...,...,...,...
6446,Isabelle de Valenzuela,42,5000000.0,2,1,Unfurnished,0.0,119047.619048
6447,Isabelle de Valenzuela,42,2100000.0,3,1,Unfurnished,0.0,50000.000000
6448,Alta Spatial,33,4999100.0,2,1,Partly furnished,0.0,151487.878788
6449,Alta Spatial,33,4051961.0,2,1,Unfurnished,0.0,122786.696970
